Publication Visuals and Analytics - BC
-----

# ----- [ CELLS ] ---------

## Define RUN TAG

In [ ]:
weather_year='2024'
run_date='20260603'
scenaio_name='BASELINE'

- Tested tweaks on Plot Adjustments

In [ ]:
MARKER_SCALE_EXISTING=0.08,
MARKER_SCALE_COMMITTED=0.08,
MARKER_HIGHLIGHT_WIDTH=0.05
ANNOTATION_ADJUSTMENT_METER:float=30E3 

- Plot and data aggregation

In [ ]:
AGGREGATION_LEVEL='Province'

- Set Root

In [ ]:
from pathlib import Path
current_dir=Path.cwd()
# Go up 2 folders
root = current_dir.parent.parent
print('Root:', root)

paper_resources=current_dir
print('Paper Resources:', paper_resources)

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

import RESource.visuals as vis
from RESource import utility as utils
from RESource.hdf5_handler import DataHandler
from RESource.CellCapacityProcessor import get_sub_nationally_aggregated_capacity
plt.style.use(root / 'RES' / 'visual_styles' / 'elsevier.mplstyle')
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)

- Load Configs

In [ ]:
RUN_ID = f'{scenaio_name.upper()}_{weather_year}_{run_date}'
config_name=f'CAN_{scenaio_name.lower()}.yaml'

In [ ]:
cfg=utils.load_config(root / 'config' / config_name)
# run_id:str=cfg.get('Scenario').get('run_id')
country_name:str=cfg.get('country')
country_kwd=country_name.replace(' ','')
region_code='BC'

In [ ]:
sub_national_unit_tag:str=cfg.get('GADM').get('datafield_mapping').get('NAME_2',None) or cfg.get('GADM').get('datafield_mapping').get('NAME_1')

CRS_m = CRS_m = cfg.get('region_mapping').get(region_code).get('CRS_meters') or cfg.get('default_CRS').get('meters')  # Default metric CRS
CRS_d = cfg.get('default_CRS').get('degrees')  # Default geographic CRS
# regions=list(cfg.get('region_mapping').keys()) #'AL','BA','XK','ME','MK','RS'
regions=['BC']
region_to_code = {
    v["name"].replace(" ", ""): k
    for k, v in cfg.get("region_mapping").items()
}

vis_save_to_root=utils.ensure_path(paper_resources / f"vis/{country_kwd}/{region_code}/{RUN_ID}")
results_save_to_root=utils.ensure_path(paper_resources / f"results/{country_kwd}/{region_code}/{RUN_ID}")

## Load Store

In [ ]:
 #All the regions should have RUN_ID results available
combined_store:dict[dict] = {}
utils.print_update(level=1,message=f"Loading data stores for {country_name} regions with RUN_ID: {RUN_ID} and Regions: {regions}")
for region in regions:
    country_dict = {}
    try:
        store = Path(root / f"data/store/{country_kwd}/{region_code}/resources_{country_kwd}_{region}_{RUN_ID}.h5")
        assert store.exists(), f"Store path doesn't exist: {store}"
        res_data = DataHandler(store, show_structure=False)
        country_dict['cells'] = res_data.from_store('cells')
        country_dict['boundary'] = res_data.from_store('boundary')
        country_dict['lines'] = res_data.from_store('lines')
        country_dict['timeseries_solar'] = res_data.from_store('timeseries/solar')
        country_dict['timeseries_wind'] = res_data.from_store('timeseries/wind')
        # country_dict['di_solar'] = res_data.from_store('dissolved_indices/solar')
        # country_dict['di_wind'] = res_data.from_store('dissolved_indices/wind')                                                                 
        # country_dict['LandAvailability'] = res_data.from_store('LandAvailability')
        combined_store[region] = country_dict
        utils.print_update(level=2,message=f"✓ Loaded data for : {cfg.get('region_mapping').get(region).get('name') if cfg.get('region_mapping').get(region) else region}.") 
    except Exception as e:
        print(f"X Error with region {region}: {e}")
        continue

## Load All Cells

In [ ]:
all_cells_dict:dict[pd.DataFrame]=[combined_store[region]['cells'] for region in combined_store]
all_cells_gdf = gpd.GeoDataFrame(pd.concat(all_cells_dict, ignore_index=False), crs=all_cells_dict[0].crs)
all_cells_gdf['ISO2']=all_cells_gdf['Province'].apply(lambda x: region_to_code[x.replace(" ","")])

- Create cells' instance for plotting (CRS-m)

In [ ]:
if all_cells_gdf.crs != CRS_m:
    all_cells_gdf_plot = all_cells_gdf.to_crs(CRS_m)
else:
    all_cells_gdf_plot = all_cells_gdf

##  Load Capacity Haircut Factors

In [ ]:
haircuts_df:pd.DataFrame=pd.read_csv(paper_resources / "data/haircuts_BC.csv")

In [ ]:
# Build dict lookups
solar_haircut_map = haircuts_df.set_index("ISO2")["solar"] / 100
wind_haircut_map  = haircuts_df.set_index("ISO2")["wind"]  / 100

In [ ]:
# Map haircut factors as explicit columns first
all_cells_gdf["solar_haircut_factor"] = all_cells_gdf["ISO2"].map(solar_haircut_map)
all_cells_gdf["wind_haircut_factor"]   = all_cells_gdf["ISO2"].map(wind_haircut_map)
# Then apply
all_cells_gdf["potential_capacity_solar_haircut"] = (
    all_cells_gdf["potential_capacity_solar"] * all_cells_gdf["solar_haircut_factor"]
)
all_cells_gdf["potential_capacity_wind_haircut"] = (
    all_cells_gdf["potential_capacity_wind"] * all_cells_gdf["wind_haircut_factor"]
)


---

# Load Validation data 

## Existing VRE sites

In [ ]:
existing_VREs_data_path=Path(root/f"data/downloaded_data/CODERS/data-pull/supply/{region_code}_wind_generators.csv")
if existing_VREs_data_path.exists():
    existing_VREs=pd.read_csv(existing_VREs_data_path)
    existing_VREs_gdf=gpd.GeoDataFrame(existing_VREs,geometry=gpd.points_from_xy(existing_VREs.longitude,existing_VREs.latitude),crs=CRS_d)
    utils.print_update(level=1,message=f"Validation data for existing VREs loaded from {existing_VREs_data_path}")
    
    existing_tech_name_mapping={
    'wind_ons':'Wind',
    'solar':'Solar'
    }
    # Create new column 'Technology' based on mapping
    existing_VREs_gdf["Technology"] = existing_VREs_gdf["gen_type"].map(existing_tech_name_mapping)

    # If some gen_type values are not in the dict, fill them with 'Unknown'
    existing_VREs_gdf["Technology"] = existing_VREs_gdf["Technology"].fillna("Unknown")

else:
    existing_VREs_gdf=None
    utils.print_warning(f"Validation data for existing VREs not found at {existing_VREs_data_path}")


if existing_VREs_gdf.crs != CRS_m:
    existing_VREs_plot = existing_VREs_gdf.to_crs(CRS_m)
else:
    existing_VREs_plot = existing_VREs_gdf

## Committed VRE Sites (BCH CFPs)

In [ ]:
committed_VREs_data_path=Path(root/"ROD_2024/BCH_CFP24.geojson")
if committed_VREs_data_path.exists():
    committed_VREs_gdf=gpd.read_file(committed_VREs_data_path, engine="fiona")
    if committed_VREs_gdf.crs is None:
        committed_VREs_gdf.set_crs(CRS_d, allow_override=True, inplace=True)
    # if committed_VREs_gdf.crs != CRS_m:
    #     committed_VREs_gdf.to_crs(CRS_m, inplace=True)
    utils.print_update(level=1,message=f"Committed VREs data loaded from {committed_VREs_data_path}")
    
    
    committed_tech_name_mapping={
        'wind':'Wind',
        'solar':'Solar'
    }
    # Create new column 'Technology' based on mapping
    committed_VREs_gdf["Technology"] = committed_VREs_gdf["resource_type"].map(existing_tech_name_mapping)

    # If some gen_type values are not in the dict, fill them with 'Unknown'
    committed_VREs_gdf["Technology"] = committed_VREs_gdf["resource_type"].fillna("Unknown")


else:
    committed_VREs_gdf=None
    utils.print_warning(f"Validation data for Committed VREs not found at {committed_VREs_data_path}")


if committed_VREs_gdf.crs != CRS_m:
    committed_VREs_plot = committed_VREs_gdf.to_crs(CRS_m)
else:
    committed_VREs_plot = committed_VREs_gdf

# Boundary

- Prepare regional boundary

In [ ]:
# Combine all region boundaries into a single GeoDataFrame
boundary_gdfs = [combined_store[region]['boundary'] for region in combined_store]
all_regions_boundary = gpd.GeoDataFrame(pd.concat(boundary_gdfs, ignore_index=True), crs=boundary_gdfs[0].crs)
all_regions_boundary_dissolved = all_regions_boundary.dissolve(by=AGGREGATION_LEVEL)[["geometry"]].reset_index()

- Process the boundary info for raster plotting

In [ ]:
all_regions_boundary_dissolved_plot=all_regions_boundary_dissolved.to_crs(CRS_m)
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = all_regions_boundary_dissolved_plot.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

# Load Capacity and Scores

## Reality Checks

### Applying LCOE thresholds 

#### Define _lcoe_ thresholds and capacity haircuts

In [ ]:
solar_lcoe_threshold=90 
wind_lcoe_threshold=130

landuse_competition_solar_share=0.50 # for example, that 50% of the land that is technically developable for solar (after all exclusions) is actually available for solar after accounting for competition with other land uses. The remaining 50% would be allocated to other uses (e.g., wind) or left undeveloped due to various constraints.
landuse_competition_wind_share=0.50 # for example, that 50% of the land that is technically developable for wind (after all exclusions) is actually available for wind after accounting for competition with other land uses. The remaining 50% would be allocated to other uses (e.g., solar) or left undeveloped due to various constraints.

#### Isolate solar and Wind Cells

In [ ]:
solar_cells= all_cells_gdf[all_cells_gdf["lcoe_solar"] <= solar_lcoe_threshold].copy()
wind_cells= all_cells_gdf[all_cells_gdf["lcoe_wind"] <= wind_lcoe_threshold].copy()

In [ ]:
capacity_cols = ["potential_capacity_solar", "potential_capacity_wind", "potential_capacity_solar_haircut", "potential_capacity_wind_haircut"]
solar_capacity_cols = capacity_cols.copy()
wind_capacity_cols = capacity_cols.copy()
solar_aggr = get_sub_nationally_aggregated_capacity(cells_with_capacity=solar_cells, capacity_cols=solar_capacity_cols, sub_national_unit_tag=AGGREGATION_LEVEL).copy()
wind_aggr = get_sub_nationally_aggregated_capacity(cells_with_capacity=wind_cells, capacity_cols=wind_capacity_cols, sub_national_unit_tag=AGGREGATION_LEVEL).copy()


# Supply Curves

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

# ── Palette ───────────────────────────────────────────────────────────────────
RESOURCE_STYLE = {
    "wind":  {"color": "#2B8CBE", "label": "Wind"},
    "solar": {"color": "#FE9929", "label": "Solar"},
}

def plot_capacity_supply_curve(
    clusters: dict,                 # {"wind": gdf, "solar": gdf}  or single gdf
    resource_type: str = None,      # required when clusters is a single gdf
    region_label: str = "",
    capacity_budget_gw: float = None,   # optional vertical line (GW)
    lcoe_ceiling: float = None,         # optional horizontal line ($/MWh)
    ax=None,
    figsize=(9, 5),
    x_limit_factor: float = 3.5,
):
    """
    Plot LCOE supply curve(s) as step functions.

    Parameters
    ----------
    clusters : dict of {resource_type: GeoDataFrame}  OR  a single GeoDataFrame
        Each GeoDataFrame must contain columns
        'lcoe_{resource_type}' and 'potential_capacity_{resource_type}'.
    resource_type : str
        Required when clusters is a single GeoDataFrame.
    region_label : str
        Appears in the title.
    capacity_budget_gw : float, optional
        Draws a vertical dashed line at this cumulative capacity (GW).
    lcoe_ceiling : float, optional
        Draws a horizontal dashed line at this LCOE ($/MWh).
    ax : matplotlib Axes, optional
        Draws into an existing Axes if provided.
    figsize : tuple

    Returns
    -------
    fig, ax
    """
    # Normalise input to dict
    if not isinstance(clusters, dict):
        assert resource_type is not None, "Provide resource_type when passing a single GeoDataFrame."
        clusters = {resource_type: clusters}

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=figsize, dpi=150)
    else:
        fig = ax.get_figure()

    legend_handles = []

    for res, gdf in clusters.items():
        cap_col  = f"potential_capacity_{res}"
        lcoe_col = f"lcoe_{res}"

        if cap_col not in gdf.columns or lcoe_col not in gdf.columns:
            print(f"WARNING: missing columns for '{res}' — skipping.")
            continue

        df = (
            gdf[[lcoe_col, cap_col]]
            .dropna()
            .sort_values(lcoe_col)
            .reset_index(drop=True)
        )

        # Cumulative capacity in GW
        cum_gw = df[cap_col].cumsum() / 1000

        style = RESOURCE_STYLE.get(res, {"color": "#555", "label": res.title()})

        # Step plot: each cluster is a horizontal segment
        ax.step(
            np.concatenate([[0], cum_gw.values]),
            np.concatenate([[df[lcoe_col].iloc[0]], df[lcoe_col].values]),
            where="post",
            color=style["color"],
            linewidth=1.8,
            zorder=3,
        )
        # Cluster dots at step origins
        ax.scatter(
            cum_gw.values,
            df[lcoe_col].values,
            color=style["color"],
            s=18,
            zorder=4,
            linewidths=0,
        )

        legend_handles.append(
            Line2D([0],[0], color=style["color"], lw=2,
                   marker='o', markersize=5, label=style["label"])
        )

    # ── Reference lines ───────────────────────────────────────────────────────
    if capacity_budget_gw is not None:
        ax.axvline(capacity_budget_gw, color="#e63946", lw=1.2,
                   ls="--", zorder=2, label=f"Budget ({capacity_budget_gw} GW)")
        legend_handles.append(
            Line2D([0],[0], color="#e63946", lw=1.2, ls="--",
                   label=f"Capacity budget ({capacity_budget_gw} GW)")
        )

    if lcoe_ceiling is not None:
        ax.axhline(lcoe_ceiling, color="#6a994e", lw=1.2,
                   ls="--", zorder=2)
        legend_handles.append(
            Line2D([0],[0], color="#6a994e", lw=1.2, ls="--",
                   label=f"LCOE ceiling ({lcoe_ceiling} $/MWh)")
        )

    # ── Axes formatting ───────────────────────────────────────────────────────
    ax.set_xlabel("Cumulative capacity  (GW)", fontsize=11)
    ax.set_ylabel("Screening Level proxy-LCOE  ($/MWh)",             fontsize=11)
    ax.set_title(
        f"Capacity Supply Curve — {region_label}" if region_label else "Capacity Supply Curve",
        fontsize=13, fontweight="bold", pad=10,
    )

    ax.xaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax.tick_params(axis="both", which="major", labelsize=10)
    ax.tick_params(axis="both", which="minor", length=3)

    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["left","bottom"]].set_linewidth(0.8)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.5, zorder=0)
    # ── Axes formatting ───────────────────────────────────────────────────────
    ax.set_xlim(left=0)   # ← replace this line with:

    if capacity_budget_gw is not None:
        ax.set_xlim(0, capacity_budget_gw * x_limit_factor)
    else:
        ax.set_xlim(left=0)
    ax.legend(
        handles=legend_handles,
        fontsize=10,
        framealpha=0.9,
        edgecolor="#cccccc",
        fancybox=False,
        loc="lower right",
    )

    if standalone:
        plt.tight_layout()
        return fig, ax
    return fig, ax

In [ ]:
def plot_energy_supply_curve(
    clusters: dict,
    resource_type: str = None,
    region_label: str = "",
    hours_per_year: int = 8760,
    energy_unit: str = "TWh",          # "TWh" or "GWh"
    lcoe_ceiling: float = None,
    energy_budget_twh: float = None,
    ax=None,
    figsize=(9, 5),
    x_limit_factor: float = 3,
):
    """
    Plot LCOE energy supply curve — X axis is cumulative annual energy output.

    Annual energy per cluster = potential_capacity (MW)
                                × {resource}_CF_mean
                                × hours_per_year   →  MWh  →  TWh or GWh

    Parameters
    ----------
    clusters : dict {resource_type: GeoDataFrame} or single GeoDataFrame
    resource_type : str   required when clusters is a single GeoDataFrame
    region_label : str
    hours_per_year : int  8760 standard; 8784 for leap years
    energy_unit : "TWh" or "GWh"
    lcoe_ceiling : float  horizontal reference line ($/MWh)
    energy_budget_twh : float  vertical reference line (TWh)
    """
    if not isinstance(clusters, dict):
        assert resource_type is not None
        clusters = {resource_type: clusters}

    divisor = {"TWh": 1e6, "GWh": 1e3}[energy_unit]

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=figsize, dpi=150)
    else:
        fig = ax.get_figure()

    legend_handles = []

    for res, gdf in clusters.items():
        cap_col  = f"potential_capacity_{res}"
        lcoe_col = f"lcoe_{res}"
        cf_col   = f"{res}_CF_mean"

        missing = [c for c in [cap_col, lcoe_col, cf_col] if c not in gdf.columns]
        if missing:
            print(f"WARNING: '{res}' missing columns {missing} — skipping.")
            continue

        df = (
            gdf[[lcoe_col, cap_col, cf_col]]
            .dropna()
            .sort_values(lcoe_col)
            .reset_index(drop=True)
        )

        # Annual energy in chosen unit
        df["_energy"] = df[cap_col] * df[cf_col] * hours_per_year / divisor
        cum_energy    = df["_energy"].cumsum()

        style = RESOURCE_STYLE.get(res, {"color": "#555", "label": res.title()})

        ax.step(
            np.concatenate([[0], cum_energy.values]),
            np.concatenate([[df[lcoe_col].iloc[0]], df[lcoe_col].values]),
            where="post",
            color=style["color"],
            linewidth=1.8,
            zorder=3,
        )
        ax.scatter(
            cum_energy.values,
            df[lcoe_col].values,
            color=style["color"],
            s=18,
            zorder=4,
            linewidths=0,
        )

        # Annotate total energy at end of curve
        ax.annotate(
            f"{cum_energy.iloc[-1]:.1f} {energy_unit}",
            xy=(cum_energy.iloc[-1], df[lcoe_col].iloc[-1]),
            xytext=(6, 0), textcoords="offset points",
            fontsize=8, color=style["color"], va="center",
        )

        legend_handles.append(
            Line2D([0],[0], color=style["color"], lw=2,
                   marker='o', markersize=5, label=style["label"])
        )

    # ── Reference lines ───────────────────────────────────────────────────────
    if energy_budget_twh is not None:
        budget_label = f"{energy_budget_twh} {energy_unit}"
        ax.axvline(energy_budget_twh, color="#e63946", lw=1.2, ls="--", zorder=2)
        legend_handles.append(
            Line2D([0],[0], color="#e63946", lw=1.2, ls="--",
                   label=f"Energy budget ({budget_label})")
        )

    if lcoe_ceiling is not None:
        ax.axhline(lcoe_ceiling, color="#6a994e", lw=1.2, ls="--", zorder=2)
        legend_handles.append(
            Line2D([0],[0], color="#6a994e", lw=1.2, ls="--",
                   label=f"LCOE ceiling ({lcoe_ceiling} $/MWh)")
        )

    # ── Axes formatting ───────────────────────────────────────────────────────
    ax.set_xlabel(f"Cumulative annual energy  ({energy_unit})", fontsize=11)
    ax.set_ylabel("Screening Level proxy-LCOE  ($/MWh)",                              fontsize=11)
    ax.set_title(
        f"Energy Supply Curve — {region_label}" if region_label else "Energy Supply Curve",
        fontsize=13, fontweight="bold", pad=10,
    )

    ax.xaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax.tick_params(axis="both", which="major", labelsize=10)
    ax.tick_params(axis="both", which="minor", length=3)
    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["left","bottom"]].set_linewidth(0.8)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.5, zorder=0)
    # ── Axes formatting ───────────────────────────────────────────────────────
    ax.set_xlim(left=0)   # ← replace this line with:

    if energy_budget_twh is not None:
        ax.set_xlim(0, energy_budget_twh * x_limit_factor)
    else:
        ax.set_xlim(left=0)
    ax.legend(
        handles=legend_handles,
        fontsize=10, framealpha=0.9,
        edgecolor="#cccccc", fancybox=False,
        loc="lower right",
    )

    if standalone:
        plt.tight_layout()
        return fig, ax
    return fig, ax

In [ ]:
# # # Single resource
# # fig, ax = plot_energy_supply_curve(
# #     clusters           = wind_cells,
# #     resource_type      = "wind",
# #     region_label       = "BC — 2024",
# #     energy_unit        = "TWh",
# #     lcoe_ceiling       = 120,
# #     energy_budget_twh  = 50,
# # )

# # Both resources
# fig, ax = plot_energy_supply_curve(
#     clusters           = {"wind": wind_cells, "solar": solar_cells},
#     region_label       = f"Weather Year {weather_year}",
#     energy_unit        = "TWh",
#     lcoe_ceiling       = 100,
#     energy_budget_twh = 10,
#     x_limit_factor = 3
# )

In [ ]:
STYLE = {
    'Baseline'        : {'color': "#56D4D8", 'ls': '-', 'lw': 5, 'alpha': 0.6},
    'Policy (restricted lands)'          : {'color': '#E31A1C', 'ls': '-', 'lw': 1.4, 'alpha': 1},
    'No Buffers'    : {'color': "#6516CC", 'ls': '--', 'lw': 1.4, 'alpha': 1},
    'Baseline - Grid Restricted (within 50 km)': {'color': '#FF7F00', 'ls': '-.', 'lw': 1.4, 'alpha': 1},
}

def supply_curve(ax, cells, tech, label):
    lcoe_col, cap_col = f'lcoe_{tech}', f'potential_capacity_{tech}'
    if lcoe_col not in cells.columns or cap_col not in cells.columns:
        return
    d = cells[[lcoe_col, cap_col]].dropna().sort_values(lcoe_col)
    x = np.cumsum(d[cap_col].values) / 1e3  # GW
    st = STYLE.get(label, {'color': 'grey', 'ls': '-','lw': 1.4, 'alpha': 0.9})
    ax.step(x, d[lcoe_col].values, where='post', lw=st['lw'],
            color=st['color'], ls=st['ls'], alpha=st['alpha'], label=label)

def energy_supply_curve(ax, cells, tech, label):
    lcoe_col = f'lcoe_{tech}'
    cap_col  = f'potential_capacity_{tech}'
    cf_col   = f'{tech}_CF_mean'
    need = [lcoe_col, cap_col, cf_col]
    if any(c not in cells.columns for c in need):
        print(f'  skip {label}/{tech}: missing {[c for c in need if c not in cells.columns]}')
        return
    d = cells[need].dropna().sort_values(lcoe_col)
    energy_twh = d[cap_col].values * d[cf_col].values * HOURS_PER_YEAR / 1e6  # MWh -> TWh
    x = np.cumsum(energy_twh)
    st = STYLE.get(label, {'color': 'grey', 'ls': '-', 'lw': 1.4, 'alpha': 0.9})
    ax.step(x, d[lcoe_col].values, where='post', lw=st['lw'],
            color=st['color'], ls=st['ls'], alpha=st['alpha'], label=label)

In [ ]:

lcoe_ceiling=90
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), dpi=150)
lcoe_ceiling=100
plot_capacity_supply_curve(
    clusters={"wind": wind_cells, "solar": solar_cells},
    region_label=f"(Weather year {weather_year})",
    capacity_budget_gw=10,
    lcoe_ceiling=lcoe_ceiling,
    ax=ax1,
    x_limit_factor=4,
)
plot_energy_supply_curve(
    clusters={"wind": wind_cells, "solar": solar_cells},
    region_label=f"(Weather year {weather_year})",
    energy_unit="TWh",
    lcoe_ceiling=lcoe_ceiling,
    energy_budget_twh=10,
    ax=ax2,
    x_limit_factor=4
)

plt.tight_layout()
fig.savefig(vis_save_to_root / "supply_curves_BC_2024.tiff", dpi=300, bbox_inches="tight")

In [ ]:
STYLE = {
    'Baseline'        : {'color': "#56D4D8", 'ls': '-', 'lw': 5, 'alpha': 0.6},
    'Policy (restricted lands)'          : {'color': '#E31A1C', 'ls': '-', 'lw': 1.4, 'alpha': 1},
    'No Buffers'    : {'color': "#6516CC", 'ls': '--', 'lw': 1.4, 'alpha': 1},
    'Baseline - Grid Restricted (within 50 km)': {'color': '#FF7F00', 'ls': '-.', 'lw': 1.4, 'alpha': 1},
}
HOURS_PER_YEAR = 8760
def supply_curve(ax, cells, tech, label):
    lcoe_col, cap_col = f'lcoe_{tech}', f'potential_capacity_{tech}'
    if lcoe_col not in cells.columns or cap_col not in cells.columns:
        return
    d = cells[[lcoe_col, cap_col]].dropna().sort_values(lcoe_col)
    x = np.cumsum(d[cap_col].values) / 1e3  # GW
    st = STYLE.get(label, {'color': 'grey', 'ls': '-','lw': 1.4, 'alpha': 0.9})
    ax.step(x, d[lcoe_col].values, where='post', lw=st['lw'],
            color=st['color'], ls=st['ls'], alpha=st['alpha'], label=label)

def energy_supply_curve(ax, cells, tech, label):
    lcoe_col = f'lcoe_{tech}'
    cap_col  = f'potential_capacity_{tech}'
    cf_col   = f'{tech}_CF_mean'
    need = [lcoe_col, cap_col, cf_col]
    if any(c not in cells.columns for c in need):
        print(f'  skip {label}/{tech}: missing {[c for c in need if c not in cells.columns]}')
        return
    d = cells[need].dropna().sort_values(lcoe_col)
    energy_twh = d[cap_col].values * d[cf_col].values * HOURS_PER_YEAR / 1e6  # MWh -> TWh
    x = np.cumsum(energy_twh)
    st = STYLE.get(label, {'color': 'grey', 'ls': '-', 'lw': 1.4, 'alpha': 0.9})
    ax.step(x, d[lcoe_col].values, where='post', lw=st['lw'],
            color=st['color'], ls=st['ls'], alpha=st['alpha'], label=label)

#### Define plot func

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.font_manager import FontProperties

def plot_resources_and_capacity_with_lcoe_thresholds(
    all_cells_gdf: gpd.GeoDataFrame=None,
    solar_cells: gpd.GeoDataFrame=None,
    wind_cells: gpd.GeoDataFrame=None,
    solar_lcoe_threshold: float | None=None,
    wind_lcoe_threshold: float | None=None,
    show_haircut_capacity: bool = False,
    show_landuse_competition_capacity: bool = False,
    figsize: tuple = (12, 6),
    dpi: int = 600,
    save_path: str | None = None,
    transparent: bool = False
):
    """
    Plot solar and wind resource maps side by side with:
    - optional LCOE threshold filtering
    - optional capacity haircut factors
    - existing VRE overlays
    - country-level aggregated capacity annotations

    Assumes the following external objects/functions already exist:
    - CRS_m
    - vis.get_data_in_map_plot(...)
    - vis.get_existing_committed_VRE_plot(...)
    - get_sub_nationally_aggregated_capacity(...)
    - existing_VREs_plot
    - all_regions_boundary_dissolved
    - utils.print_warning(...)
    """
    if solar_cells is None or wind_cells is None:
        all_cells_gdf = all_cells_gdf.copy()
        if all_cells_gdf.empty:
            raise ValueError("all_cells_gdf is empty, cannot use as default for solar_cells or wind_cells.")
        if solar_cells is None:
            solar_cells = all_cells_gdf.copy()
        if wind_cells is None:
            wind_cells = all_cells_gdf.copy()
    # -------------------------------------------------
    # Validate required columns
    # -------------------------------------------------'
    capacity_cols = ["potential_capacity_solar", "potential_capacity_wind"]
    solar_capacity_cols = capacity_cols.copy()
    wind_capacity_cols = capacity_cols.copy()
   
    
    if show_landuse_competition_capacity:
        solar_capacity_col = "total_capacity_solar_afterLandUseCompetition"
        wind_capacity_col  = "total_capacity_wind_afterLandUseCompetition"
        solar_capacity_cols.extend([f"{solar_capacity_col}"])
        wind_capacity_cols.extend([f"{wind_capacity_col}"])
    elif show_haircut_capacity:
        solar_capacity_col = "potential_capacity_solar_haircut"
        wind_capacity_col  = "potential_capacity_wind_haircut"
        solar_capacity_cols.extend([f"{solar_capacity_col}"])
        wind_capacity_cols.extend([f"{wind_capacity_col}"])
    else:
        solar_capacity_col = "potential_capacity_solar"
        wind_capacity_col  = "potential_capacity_wind"
    

    if solar_cells.empty:
        raise ValueError("solar_cells is empty.")
    if wind_cells.empty:
        raise ValueError("wind_cells is empty.")

    # -------------------------------------------------
    # CRS harmonization
    # -------------------------------------------------
    if solar_cells.crs != CRS_m:
        solar_cells_plot = solar_cells.to_crs(CRS_m)
    else:
        solar_cells_plot = solar_cells.copy()

    if wind_cells.crs != CRS_m:
        wind_cells_plot = wind_cells.to_crs(CRS_m)
    else:
        wind_cells_plot = wind_cells.copy()

    if all_regions_boundary_dissolved.crs != CRS_m:
        boundaries_plot = all_regions_boundary_dissolved.to_crs(CRS_m)
    else:
        boundaries_plot = all_regions_boundary_dissolved.copy()

    if existing_VREs_plot.crs != CRS_m:
        existing_vres_plot = existing_VREs_plot.to_crs(CRS_m)
    else:
        existing_vres_plot = existing_VREs_plot.copy()
    
    if committed_VREs_plot is not None and committed_VREs_plot.crs != CRS_m:
        committed_vres_plot = committed_VREs_plot.to_crs(CRS_m)
    else:
        committed_vres_plot = committed_VREs_plot.copy()

    # -------------------------------------------------
    # Threshold filtering
    # -------------------------------------------------
    if solar_lcoe_threshold is not None:
        cells_thresholded_solar = solar_cells_plot.loc[
            solar_cells_plot["lcoe_solar"] <= solar_lcoe_threshold
        ].copy()
    else:
        utils.print_warning("No solar LCOE threshold provided; plotting all cells for solar.")
        cells_thresholded_solar = solar_cells_plot.copy()

    if wind_lcoe_threshold is not None:
        cells_thresholded_wind = wind_cells_plot.loc[
            wind_cells_plot["lcoe_wind"] <= wind_lcoe_threshold
        ].copy()
    else:
        utils.print_warning("No wind LCOE threshold provided; plotting all cells for wind.")
        cells_thresholded_wind = wind_cells_plot.copy()

    # -------------------------------------------------
    # Create figure
    # -------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize, dpi=dpi)

    # -------------------------------------------------
    # Gray base first: all cells
    # This becomes the "excluded / not selected" background
    # -------------------------------------------------
    solar_cells_plot.plot(ax=ax1, color="grey", edgecolor="none", alpha=0.7, zorder=1)
    wind_cells_plot.plot(ax=ax2, color="grey", edgecolor="none", alpha=0.7, zorder=1)
    if all_cells_gdf_plot is not None and not all_cells_gdf_plot.empty:
        all_cells_gdf_plot.plot(ax=ax1, color="grey", edgecolor="none", alpha=0.7, zorder=1)
        all_cells_gdf_plot.plot(ax=ax2, color="grey", edgecolor="none", alpha=0.7, zorder=1)

    # -------------------------------------------------
    # Resource maps on top
    # -------------------------------------------------
    if not cells_thresholded_solar.empty:
        vis.get_data_in_map_plot(
            cells_thresholded_solar,
            resource_type="solar",
            datafield="score",
            compass_size=12,
            ax=ax1,
            score_threshold=250,
            show=False,
        )

    if not cells_thresholded_wind.empty:
        vis.get_data_in_map_plot(
            cells_thresholded_wind,
            resource_type="wind",
            datafield="score",
            compass_size=12,
            ax=ax2,
            score_threshold=250,
            show=False,
        )

    no_land_patch = mpatches.Patch(
        facecolor="lightgray",
        edgecolor="gray",
        alpha=0.7,
        label="Economically unfeasible or no developable land",
    )

    # -------------------------------------------------
    # Existing solar / wind overlays
    # -------------------------------------------------
    solar_legends = []
    wind_legends = []

    if "Technology" not in existing_vres_plot.columns:
        raise ValueError("existing_VREs_plot must contain a 'Technology' column.")

    existing_vres_solar = existing_vres_plot.loc[
        existing_vres_plot["Technology"].astype(str).str.lower() == "solar"
    ].copy()

    existing_vres_wind = existing_vres_plot.loc[
        existing_vres_plot["Technology"].astype(str).str.lower() == "wind"
    ].copy()
        
    # -------------------------------------------------
    # Committed solar / wind overlays
    # -------------------------------------------------

    if "Technology" not in committed_vres_plot.columns:
        raise ValueError("committed_VREs_plot must contain a 'Technology' column.")

    committed_vres_solar = committed_vres_plot.loc[
        committed_vres_plot["Technology"].astype(str).str.lower() == "solar"
    ].copy()

    committed_vres_wind = committed_vres_plot.loc[
        committed_vres_plot["Technology"].astype(str).str.lower() == "wind"
    ].copy()

    # if not committed_vres_solar.empty:
    ax1, solar_legends = vis.get_existing_committed_VRE_plot(
            ax=ax1,
            existing_VREs_gdf=existing_vres_solar,
            existing_VRE_type_column="Technology",
            committed_VREs_gdf=committed_vres_solar,
            committed_VRE_type_column="Technology",
            target_crs=CRS_m,
            marker_scale_existing=MARKER_SCALE_EXISTING,
            marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
            marker_scale_committed=MARKER_SCALE_COMMITTED,
        )

    # if not committed_vres_wind.empty:
    ax2, wind_legends = vis.get_existing_committed_VRE_plot(
            ax=ax2,
            existing_VREs_gdf=existing_vres_wind,
            existing_VRE_type_column="Technology",
            committed_VREs_gdf=committed_vres_wind,
            committed_VRE_type_column="Technology",
            target_crs=CRS_m,
            marker_scale_existing=MARKER_SCALE_EXISTING,
            marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
            marker_scale_committed=MARKER_SCALE_COMMITTED,
        )


    # -------------------------------------------------
    # Aggregated capacity using thresholded cells
    # IMPORTANT: use filtered subsets, not all cells
    # -------------------------------------------------

    solar_aggr = get_sub_nationally_aggregated_capacity(cells_with_capacity=cells_thresholded_solar, capacity_cols=solar_capacity_cols, sub_national_unit_tag=AGGREGATION_LEVEL).copy()
    wind_aggr = get_sub_nationally_aggregated_capacity(cells_with_capacity=cells_thresholded_wind, capacity_cols=wind_capacity_cols, sub_national_unit_tag=AGGREGATION_LEVEL).copy()


    # Convert MW -> GW if your source is MW
    ## Conditional Aggregation based on func args

    solar_aggr[f"{solar_capacity_col}_GW"] = (
        solar_aggr[f"{solar_capacity_col}"]  / 1e3
    )
    wind_aggr[f"{wind_capacity_col}_GW"] = (
        wind_aggr[f"{wind_capacity_col}"]/ 1e3
    )

    # Merge to boundary map for geometry consistency
    solar_map = boundaries_plot.merge(
        solar_aggr[[AGGREGATION_LEVEL, f"{solar_capacity_col}_GW"]],
        on=AGGREGATION_LEVEL,
        how="left",
    )
    wind_map = boundaries_plot.merge(
        wind_aggr[[AGGREGATION_LEVEL, f"{wind_capacity_col}_GW"]],
        on=AGGREGATION_LEVEL,
        how="left",
    )

    # Country boundaries
    solar_map.boundary.plot(ax=ax1, color="black", linewidth=0.7,alpha=0.7, zorder=6)
    wind_map.boundary.plot(ax=ax2, color="black", linewidth=0.7,alpha=0.7, zorder=6)

    # -------------------------------------------------
    # Country labels + capacities
    # representative_point() is safer than centroid
    # -------------------------------------------------
#
    Country_name_Y_adjustment = ANNOTATION_ADJUSTMENT_METER

    for _, row in solar_map.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            continue

        cap = row.get(f"{solar_capacity_col}_GW")
        
        if cap >0:
            label_point = row.geometry.representative_point()
            x, y = label_point.x, label_point.y

            ax1.annotate(
                f"{cap:.1f}",
                (x, y),
                color="black",
                fontsize=10,
                ha="center",
                va="center",
                fontweight="bold",
                zorder=7,
                path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.7)],
            )
            ax1.annotate(
                row[AGGREGATION_LEVEL],
                (x, y + Country_name_Y_adjustment),
                color="black",
                fontsize=9,
                ha="center",
                va="bottom",
                zorder=7,
                path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.7)],
            )
        else:
            utils.print_warning(f"Skipping label for {row[AGGREGATION_LEVEL]} in solar map due to non-positive capacity: {cap} GW")

    for _, row in wind_map.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            continue

        cap = row.get(f"{wind_capacity_col}_GW")
        if cap >0:
            label_point = row.geometry.representative_point()
            x, y = label_point.x, label_point.y
            ax2.annotate(
                f"{cap:.1f}",
                (x, y),
                color="black",
                fontsize=10,
                ha="center",
                va="center",
                fontweight="bold",
                zorder=7,
                path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.7)],
            )
            ax2.annotate(
                row[AGGREGATION_LEVEL],
                (x, y + Country_name_Y_adjustment),
                color="black",
                fontsize=9,
                ha="center",
                va="bottom",
                zorder=7,
                path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.7)],
            )
        else:
            utils.print_warning(f"Skipping label for {row[AGGREGATION_LEVEL]} in wind map due to non-positive capacity: {cap} GW")
 

    # -------------------------------------------------
    # Axes cleanup
    # -------------------------------------------------
    ax1.set_axis_off()
    ax2.set_axis_off()

    # -------------------------------------------------
    # Shared legend
    # -------------------------------------------------
    all_legends = []
    all_legends.extend(solar_legends if solar_legends is not None else [])
    all_legends.extend(wind_legends if wind_legends is not None else [])
    all_legends.append(no_land_patch)

    # de-duplicate legend labels
    unique_handles = []
    seen_labels = set()
    for h in all_legends:
        label = h.get_label()
        if label not in seen_labels:
            unique_handles.append(h)
            seen_labels.add(label)

    fig.legend(
        handles=unique_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.90),
        ncol=1,
        fontsize=9.5,
        frameon=False,
    )

    # -------------------------------------------------
    # Plot note
    # -------------------------------------------------
    note_parts = []

    if solar_lcoe_threshold is not None:
        note_parts.append(f"Solar LCOE threshold ≤ {solar_lcoe_threshold:.1f} \\$/MWh")
    else:
        note_parts.append("Solar LCOE threshold: not applied")

    if wind_lcoe_threshold is not None:
        note_parts.append(f"Wind LCOE threshold ≤ {wind_lcoe_threshold:.1f} \\$/MWh")
    else:
        note_parts.append("Wind LCOE threshold: not applied")

    if show_haircut_capacity:
        note_parts.append("with capacity haircuts")
    else:        
        note_parts.append("without capacity haircuts")
    
    if show_landuse_competition_capacity:
        note_parts.append("with land-use competition")
    else:
        note_parts.append("no land-use competition")    
    
    plot_note = r" $\bullet$ ".join(note_parts)
    fp = FontProperties(weight='bold', size=9.5)
    fig.text(
        0.5,
        0.02,
        plot_note,
        ha="center",
        va="bottom",
        fontproperties=fp,
        bbox=dict(
            boxstyle="round,pad=0.4",
            facecolor="None",
            edgecolor="gray",
            alpha=0.9,
        ),
    )

    # -------------------------------------------------
    # Layout and save
    # -------------------------------------------------
    # fig.suptitle("VRE Resources and Capacity by Country with LCOE Thresholds", fontsize=16, fontweight="bold", y=0.98)
    plt.tight_layout(rect=[0, 0.05, 1, 0.92])

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight", transparent=transparent)

    return fig, (ax1, ax2)

### Landuse Competition in Cells

#### Get landuse competition flags on cells

In [ ]:
import geopandas as gpd
def get_landuse_competitions_flags(solar_cells:gpd.GeoDataFrame, 
                                   wind_cells:gpd.GeoDataFrame):
    
    assert "geometry" in solar_cells.columns, "solar_cells must have a 'geometry' column"
    assert "geometry" in wind_cells.columns, "wind_cells must have a 'geometry' column"
    
    if solar_cells.crs != wind_cells.crs:
        wind_cells = wind_cells.to_crs(solar_cells.crs)

    solar_cells = solar_cells.copy()
    wind_cells = wind_cells.copy()

    # Use WKB as a stable geometry key
    solar_cells["_geom_key"] = solar_cells.geometry.to_wkb()
    wind_cells["_geom_key"] = wind_cells.geometry.to_wkb()

    wind_keys = set(wind_cells["_geom_key"])
    solar_keys = set(solar_cells["_geom_key"])

    solar_cells["Land_use_competition"] = solar_cells["_geom_key"].isin(wind_keys)
    wind_cells["Land_use_competition"] = wind_cells["_geom_key"].isin(solar_keys)

    solar_cells.drop(columns="_geom_key", inplace=True)
    wind_cells.drop(columns="_geom_key", inplace=True)
    
    # Solar
    print(
        solar_cells.groupby("Land_use_competition")["potential_capacity_solar_haircut"]
        .sum()/1E3
    )

    # Wind
    print(
        wind_cells.groupby("Land_use_competition")["potential_capacity_wind_haircut"]
        .sum()/1E3
    )
    return solar_cells, wind_cells

In [ ]:
solar_cells, wind_cells = get_landuse_competitions_flags(solar_cells, wind_cells)

#### Define Landuse Intensity from config

In [ ]:
solar_lui=float(cfg.get("capacity_disaggregation").get('solar').get('landuse_intensity'))
wind_lui=float(cfg.get("capacity_disaggregation").get('wind').get('landuse_intensity'))

#### Split cells based on landuse competition flags and allocate capacity

In [ ]:
# -------------------------------
# 1. Split data
# -------------------------------
solar_cells_NO_LANDUSE_COMPETITION = solar_cells.loc[
    ~solar_cells["Land_use_competition"]
].copy()
solar_cells_LANDUSE_COMPETITION = solar_cells.loc[
    solar_cells["Land_use_competition"]
].copy()
wind_cells_LANDUSE_COMPETITION = wind_cells.loc[
    solar_cells_LANDUSE_COMPETITION.index
].copy()

# -------------------------------
# 2. Capacity allocation
# -------------------------------
solar_cells_NO_LANDUSE_COMPETITION["potential_capacity_solar_no_comp"] = (
    solar_cells_NO_LANDUSE_COMPETITION["Developable_area_solar"]
    * solar_lui
)*solar_cells_NO_LANDUSE_COMPETITION["solar_haircut_factor"]

solar_cells_LANDUSE_COMPETITION["potential_capacity_solar_40pct"] = (
    solar_cells_LANDUSE_COMPETITION["Developable_area_solar"]
    * solar_lui
    * landuse_competition_solar_share
)*solar_cells_LANDUSE_COMPETITION["solar_haircut_factor"]

wind_cells_LANDUSE_COMPETITION["potential_capacity_wind_60pct"] = (
    wind_cells_LANDUSE_COMPETITION["Developable_area_wind"]
    * wind_lui
    * landuse_competition_wind_share
)*wind_cells_LANDUSE_COMPETITION["wind_haircut_factor"]


### Aggregate capacity based on allocated capacity (competitive, non-competitive)

In [ ]:
# -------------------------------
# 3. Aggregate to country level for reporting
# -------------------------------

#-------- Solar without land use competition
solar_no_comp_cap = (
    solar_cells_NO_LANDUSE_COMPETITION
    .dissolve(by=AGGREGATION_LEVEL, aggfunc="sum")
    .reset_index()
)
total_no_comp_solar = solar_cells_NO_LANDUSE_COMPETITION["potential_capacity_solar_no_comp"].sum() / 1e3

solar_comp_cap = (
    solar_cells_LANDUSE_COMPETITION
    .dissolve(by=AGGREGATION_LEVEL, aggfunc="sum")
    .reset_index()
)

total_solar_comp = solar_cells_LANDUSE_COMPETITION["potential_capacity_solar_40pct"].sum() / 1e3

#-------- Wind with land use competition (since we assume all wind is in competition)
wind_comp_cap = (
    wind_cells_LANDUSE_COMPETITION
    .dissolve(by=AGGREGATION_LEVEL, aggfunc="sum")
    .reset_index()
)
total_wind_comp = wind_cells_LANDUSE_COMPETITION["potential_capacity_wind_60pct"].sum() / 1e3

#----------------------------------------------------------------------------------------------------------------------------
# Combine solar and wind for final reporting and visualization
#---------------------------------------------------------------------------------------------------------------------------

## For solar, we have two categories (comp and no-comp) that we want to sum together for the final total potential capacity. 

solar_combined = solar_cells.copy()
solar_combined["total_capacity_solar_afterLandUseCompetition"] = 0.0
no_comp_mask = ~solar_combined["Land_use_competition"]
solar_combined.loc[no_comp_mask, "total_capacity_solar_afterLandUseCompetition"] = (
    solar_combined.loc[no_comp_mask, "Developable_area_solar"]
    * solar_lui
)*solar_combined.loc[no_comp_mask, "solar_haircut_factor"]

comp_mask = solar_combined["Land_use_competition"]
solar_combined.loc[comp_mask, "total_capacity_solar_afterLandUseCompetition"] = (
    solar_combined.loc[comp_mask, "Developable_area_solar"]
    * solar_lui * landuse_competition_solar_share
)*solar_combined.loc[comp_mask, "solar_haircut_factor"]

solar_combined = solar_combined[solar_combined["total_capacity_solar_afterLandUseCompetition"] > 0].copy()
total_solar = solar_combined["total_capacity_solar_afterLandUseCompetition"].sum() / 1e3

## For wind, we only have the comp category. We can create combined columns in the original solar_cells and wind_cells dataframes for clarity, and then also create country-level aggregates.

wind_combined = wind_cells_LANDUSE_COMPETITION.copy()
wind_combined["total_capacity_wind_afterLandUseCompetition"] = (
    wind_combined["Developable_area_wind"]
    * wind_lui * landuse_competition_wind_share
)*wind_combined["wind_haircut_factor"]

wind_combined = wind_combined[wind_combined["total_capacity_wind_afterLandUseCompetition"] > 0].copy()

## Annotation Helper: Create country-level aggregates with geometry for plotting and annotation. We can use the original boundaries for geometry, but we need to sum the capacities by country first.
solar_country = (
    solar_combined[[AGGREGATION_LEVEL, "total_capacity_solar_afterLandUseCompetition", "geometry"]]
    .dissolve(by=AGGREGATION_LEVEL, aggfunc="sum").reset_index()
)
solar_country["centroid"] = solar_country.geometry.representative_point()

wind_country = (
    wind_combined[[AGGREGATION_LEVEL, "total_capacity_wind_afterLandUseCompetition", "geometry"]]
    .dissolve(by=AGGREGATION_LEVEL, aggfunc="sum").reset_index()
)
wind_country["centroid"] = wind_country.geometry.representative_point()

In [ ]:
solar_no_comp_cap["centroid"] = solar_no_comp_cap.geometry.representative_point()
wind_comp_cap["centroid"] = wind_comp_cap.geometry.representative_point()
solar_comp_cap["centroid"] = solar_comp_cap.geometry.representative_point()


def annotate_countries(ax, gdf, cap_col, country_col=AGGREGATION_LEVEL,
                       unit="GW", fontsize=6.5):
    for _, row in gdf.iterrows():
        pt = row["centroid"]
        cap = row[cap_col] / 1e3
        label = f"{row[country_col]}\n{cap:.2f} {unit}"
        ax.annotate(
            label,
            xy=(pt.x, pt.y),
            ha="center", va="center",
            fontsize=fontsize,
            fontweight="bold",
            color="black",
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                edgecolor="grey",
                alpha=0.8,
            ),
            zorder=5,
        )

# ===============================================================
# FIGURE 1: Three-panel split (non-competitive / competitive solar / competitive wind)
# ===============================================================
fig1, axes1 = plt.subplots(1, 3, figsize=(15, 5), dpi=200)

# --- Left panel ---
solar_cells_NO_LANDUSE_COMPETITION.plot(
    column="lcoe_solar", cmap="YlOrRd", ax=axes1[0],
    vmin=solar_cells_NO_LANDUSE_COMPETITION["lcoe_solar"].min(),
    vmax=solar_cells_NO_LANDUSE_COMPETITION["lcoe_solar"].max(),
    legend=False,
)
solar_cells_LANDUSE_COMPETITION.plot(
    ax=axes1[0], color="lightgrey", edgecolor="darkgrey",
    linewidth=0.3, alpha=0.7,
)
all_regions_boundary_dissolved.boundary.plot(ax=axes1[0], color="black", linewidth=0.6, zorder=3)
overlay_patch = mpatches.Patch(
    facecolor="lightgrey", edgecolor="darkgrey",
    label="Competitive land (not allocated here)",
)
axes1[0].legend(handles=[overlay_patch], loc="lower left", frameon=True, fontsize=7)
annotate_countries(axes1[0], solar_no_comp_cap, "potential_capacity_solar_no_comp")



axes1[0].set_title(
    f"Solar — No land-use competition\nTotal = {total_no_comp_solar:.2f} GW",
    fontsize=10, fontweight="bold",
)
axes1[0].set_axis_off()

# --- Middle panel ---
solar_cells_LANDUSE_COMPETITION.plot(
    column="lcoe_solar", cmap="YlOrRd", ax=axes1[1],
    vmin=solar_cells_LANDUSE_COMPETITION["lcoe_solar"].min(),
    vmax=solar_cells_LANDUSE_COMPETITION["lcoe_solar"].max(),
    legend=False,
)
all_regions_boundary_dissolved.boundary.plot(ax=axes1[1], color="black", linewidth=0.6, zorder=3)
annotate_countries(axes1[1], solar_comp_cap, "potential_capacity_solar_40pct")

axes1[1].set_title(
    f"Competitive land → 40 % Solar\nTotal = {total_solar_comp:.2f} GW",
    fontsize=10, fontweight="bold",
)
axes1[1].set_axis_off()

# --- Right panel ---
wind_cells_LANDUSE_COMPETITION.plot(
    column="lcoe_wind", cmap="Blues", ax=axes1[2],
    vmin=wind_cells_LANDUSE_COMPETITION["lcoe_wind"].min(),
    vmax=wind_cells_LANDUSE_COMPETITION["lcoe_wind"].max(),
    legend=False,
)
all_regions_boundary_dissolved.boundary.plot(ax=axes1[2], color="black", linewidth=0.7, zorder=3)
annotate_countries(axes1[2], wind_comp_cap, "potential_capacity_wind_60pct")

axes1[2].set_title(
    f"Competitive land → 60 % Wind\nTotal = {total_wind_comp:.2f} GW",
    fontsize=10, fontweight="bold",
)
axes1[2].set_axis_off()

# Colorbars
sm1 = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(
    vmin=solar_cells_NO_LANDUSE_COMPETITION["lcoe_solar"].min(),
    vmax=solar_cells_NO_LANDUSE_COMPETITION["lcoe_solar"].max(),
))
sm1._A = []
fig1.colorbar(sm1, ax=axes1[:2], orientation="horizontal", fraction=0.04, pad=0.08
              ).set_label("LCOE (USD/MWh)", fontsize=9)

sm1w = plt.cm.ScalarMappable(cmap="Blues", norm=plt.Normalize(
    vmin=wind_cells_LANDUSE_COMPETITION["lcoe_wind"].min(),
    vmax=wind_cells_LANDUSE_COMPETITION["lcoe_wind"].max(),
))
sm1w._A = []
fig1.colorbar(sm1w, ax=axes1[2], orientation="horizontal", fraction=0.04, pad=0.08
              ).set_label("LCOE (USD/MWh)", fontsize=9)

plt.tight_layout()
fig1.savefig(vis_save_to_root/"Land_use_competition_solar_wind_partial.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()

# ===============================================================
# FIGURE 2: Two-panel summary (total solar / total wind)
# ===============================================================


fig2, axes2 = plt.subplots(1, 2, figsize=(10, 6), dpi=200)
if all_cells_gdf is not None and not all_cells_gdf.empty:
    all_cells_gdf.plot(ax=axes2[0], color="grey", edgecolor="none", alpha=0.7, zorder=1)
    all_cells_gdf.plot(ax=axes2[1], color="grey", edgecolor="none", alpha=0.7, zorder=1)
    
overlay_patch = mpatches.Patch(
    facecolor="lightgrey", edgecolor="darkgrey",
    label="Economically Infeasible or non-developable land",
)
axes2[0].legend(handles=[overlay_patch], loc="lower right", frameon=True, fontsize=7)   

# --- Left: Total Solar ---
solar_combined.plot(
    column="lcoe_solar", cmap="YlOrRd", ax=axes2[0],
    vmin=solar_combined["lcoe_solar"].min(),
    vmax=solar_combined["lcoe_solar"].max(),
    legend=False, linewidth=0.1, edgecolor="face",
)
all_regions_boundary_dissolved.boundary.plot(ax=axes2[0], color="black", linewidth=0.7, zorder=3)
annotate_countries(axes2[0], solar_country, "total_capacity_solar_afterLandUseCompetition")

axes2[0].set_title(
    f"Total Solar Potential (non-competitive 100 % + competitive {landuse_competition_solar_share * 100:.0f} %)\n"
    f"Total = {total_solar:.2f} GW",
    fontsize=10, fontweight="bold",
)
axes2[0].set_axis_off()

# --- Right: Total Wind ---
wind_combined.plot(
    column="lcoe_wind", cmap="Blues", ax=axes2[1],
    vmin=wind_combined["lcoe_wind"].min(),
    vmax=wind_combined["lcoe_wind"].max(),
    legend=False, linewidth=0.1, edgecolor="face",
)
all_regions_boundary_dissolved.boundary.plot(ax=axes2[1], color="black", linewidth=0.7, zorder=3)
annotate_countries(axes2[1], wind_country, "total_capacity_wind_afterLandUseCompetition")
total_wind = wind_combined["total_capacity_wind_afterLandUseCompetition"].sum() / 1e3
axes2[1].set_title(
    f"Total Wind Potential (competitive land {landuse_competition_wind_share * 100:.0f} %)\n"
    f"Total = {total_wind:.2f} GW",
    fontsize=10, fontweight="bold",
)
axes2[1].set_axis_off()

# Colorbars
sm2s = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(
    vmin=solar_combined["lcoe_solar"].min(),
    vmax=solar_combined["lcoe_solar"].max(),
))
sm2s._A = []
fig2.colorbar(sm2s, ax=axes2[0], orientation="horizontal", fraction=0.04, pad=0.06
              ).set_label("Solar LCOE (USD/MWh)", fontsize=9)

sm2w = plt.cm.ScalarMappable(cmap="Blues", norm=plt.Normalize(
    vmin=wind_combined["lcoe_wind"].min(),
    vmax=wind_combined["lcoe_wind"].max(),
))
sm2w._A = []
fig2.colorbar(sm2w, ax=axes2[1], orientation="horizontal", fraction=0.04, pad=0.06
              ).set_label("Wind LCOE (USD/MWh)", fontsize=9)

plt.tight_layout()
fig2.savefig(vis_save_to_root/"Landuse_competition_total_solar_wind_potential_by_country.png", dpi=300, bbox_inches="tight",transparent=True)
plt.show()

## Visualize the comparisons

### Capacity WITHOUT - LCOE thresholds and haircut factors

In [ ]:
utils.print_banner("Plotting resources and capacity WITHOUT LCOE thresholds and WITHOUT haircut factors")
# =========================================================
# Plot with threshold but WITHOUT haircut
# =========================================================
plot_resources_and_capacity_with_lcoe_thresholds(
    all_cells_gdf=all_cells_gdf_plot,
    # solar_cells=solar_cells,
    # wind_cells=wind_cells,
    show_haircut_capacity=False,
    show_landuse_competition_capacity=False,
    solar_lcoe_threshold=None,
    wind_lcoe_threshold=None,
    save_path=vis_save_to_root / "Capacity_without_LCOE_Thresholds_WITHOUT_HAIRCUT_WITHOUT_LUC.png",
    figsize=(12,6),
    dpi=1000,
    transparent=True,
)

### capacity WITH LCOE thresholds but WITHOUT haircut factors

In [ ]:
utils.print_banner("Plotting resources and capacity WITH LCOE thresholds but WITHOUT haircut factors")
# =========================================================
# Plot with threshold but WITHOUT haircut
# =========================================================
plot_resources_and_capacity_with_lcoe_thresholds(
    all_cells_gdf=None,
    solar_cells=solar_combined,
    wind_cells=wind_combined,
    show_haircut_capacity=False,
    show_landuse_competition_capacity=False,
    solar_lcoe_threshold=solar_lcoe_threshold,
    wind_lcoe_threshold=wind_lcoe_threshold,
    save_path=vis_save_to_root / "Capacity_with_LCOE_Thresholds_WITHOUT_HAIRCUT_WITHOUT_LUC.png",
    figsize=(12,8),
    dpi=1000,
    transparent=True,
)

In [ ]:
utils.print_banner("Plotting resources and capacity WITH LCOE thresholds but WITHOUT haircut factors")
# =========================================================
# Plot with threshold but WITHOUT haircut
# =========================================================
plot_resources_and_capacity_with_lcoe_thresholds(
    all_cells_gdf=None,
    solar_cells=solar_combined,
    wind_cells=wind_combined,
    show_haircut_capacity=True,
    show_landuse_competition_capacity=False,
    solar_lcoe_threshold=solar_lcoe_threshold,
    wind_lcoe_threshold=wind_lcoe_threshold,
    save_path=vis_save_to_root / "Capacity_with_LCOE_Thresholds_WITH_HAIRCUT_WITHOUT_LUC.png",
    figsize=(12,6),
    dpi=500,
    transparent=True,
)

### Plot landuse competition results

In [ ]:
utils.print_banner("Plotting resources and capacity WITH LCOE thresholds but WITHOUT haircut factors")
# =========================================================
# Plot with threshold but WITHOUT haircut
# =========================================================
plot_resources_and_capacity_with_lcoe_thresholds(
    all_cells_gdf=all_cells_gdf,
    solar_cells=solar_combined,  # use the combined solar with competition adjustments
    wind_cells=wind_combined,  # use the combined wind with competition adjustments
    show_haircut_capacity=True,
    show_landuse_competition_capacity=True,
    solar_lcoe_threshold=solar_lcoe_threshold,
    wind_lcoe_threshold=wind_lcoe_threshold,
    save_path=vis_save_to_root / "Capacity_with_LCOE_Thresholds_with_HAIRCUT_with_LUC.png",
    figsize=(12,6),
    dpi=500,
    transparent=True,
)

# Attributs' Map

In [ ]:
AGGREGATION_LEVEL_2='Region'

- Calculate Country Aggregated data

In [ ]:
solar_cells_aggr = get_sub_nationally_aggregated_capacity(cells_with_capacity=solar_combined, capacity_cols=['total_capacity_solar_afterLandUseCompetition'], sub_national_unit_tag=AGGREGATION_LEVEL_2)
wind_cells_aggr = get_sub_nationally_aggregated_capacity(cells_with_capacity=wind_combined, capacity_cols=['total_capacity_wind_afterLandUseCompetition'], sub_national_unit_tag=AGGREGATION_LEVEL_2)

In [ ]:
merged = solar_cells_aggr[[AGGREGATION_LEVEL_2, "total_capacity_solar_afterLandUseCompetition"]].merge(
    wind_cells_aggr[[AGGREGATION_LEVEL_2, "total_capacity_wind_afterLandUseCompetition"]],
        left_on=AGGREGATION_LEVEL_2, right_on=AGGREGATION_LEVEL_2
)

In [ ]:
all_regions_boundary_dissolved = all_regions_boundary.dissolve(by=AGGREGATION_LEVEL_2)[["geometry"]].reset_index()
all_regions_boundary_dissolved_plot=all_regions_boundary_dissolved.to_crs(CRS_m)
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = all_regions_boundary_dissolved_plot.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

In [ ]:
aggregated_map = all_regions_boundary_dissolved_plot.merge(
    merged[
        [
            AGGREGATION_LEVEL_2,
            "total_capacity_solar_afterLandUseCompetition",
            "total_capacity_wind_afterLandUseCompetition",
        ]
    ],
    on=AGGREGATION_LEVEL_2,
    how="left",
)


- Add calculated data

In [ ]:
aggregated_map["potential_capacity_solar_GW"] = aggregated_map["total_capacity_solar_afterLandUseCompetition"] / 1e3
aggregated_map["potential_capacity_wind_GW"] = aggregated_map["total_capacity_wind_afterLandUseCompetition"] / 1e3

- plot

In [ ]:
if aggregated_map.crs != CRS_m:
    aggregated_map_plot = aggregated_map.to_crs(CRS_m)
else:
    aggregated_map_plot = aggregated_map

### Aggregated Capacity

In [ ]:
aggregated_map_plot=aggregated_map_plot.fillna(0)

- Calculate

In [ ]:
import matplotlib.patheffects as pe

# ========= Create subplots =========
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 8), dpi=700)
Country_name_Y_adjustment:float=30E3 #in meters for CRS_m


# ========= Plot solar capacity =========
aggregated_map_plot.plot(
    column="potential_capacity_solar_GW",
    cmap="Reds",
    linewidth=0.8,
    edgecolor="k",
    legend=True,
    ax=ax1,
    legend_kwds={"label": "Solar Potential (GW)", "shrink": 0.5}
)
ax1.set_title("Solar Potential Capacity (GW)", fontsize=12, weight="bold")
ax1.set_axis_off()

# ========= Annotate numbers and country names with white halo =========
for idx, row in aggregated_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_solar_GW"]
    if val == 0:
        continue  # Skip annotation for zero values
    else:   
        # Capacity value
        ax1.annotate(
            f"{val:,.2f}",
            (centroid.x, centroid.y),
            color="black",
            fontsize=11,
            ha="center",
            va="center",
            fontweight="bold",
            path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
        )
        if val >= 0.5:
            # ========= Country name slightly above =========
            ax1.annotate(
                row[AGGREGATION_LEVEL_2],
                (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
                color="black",
                fontsize=9,
                ha="center",
                va="bottom",
                fontweight="normal",
                path_effects=[pe.withStroke(linewidth=2, foreground="white",alpha=0.6)]
            )

# ========= Plot wind capacity ==============
aggregated_map_plot.plot(
    column="potential_capacity_wind_GW",
    cmap="Blues",
    linewidth=0.8,
    edgecolor="k",
    legend=True,
    ax=ax2,
    legend_kwds={"label": "Wind Potential (GW)", "shrink": 0.5}
)
ax2.set_title("Wind Potential Capacity (GW)", fontsize=12, weight="bold")
ax2.set_axis_off()

# ========= Annotate numbers and country names with white halo =========
for idx, row in aggregated_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_wind_GW"]
    if val == 0:
        continue  # Skip annotation for zero values
    else:
        # Capacity value
        ax2.annotate(
            f"{val:,.2f}",
            (centroid.x, centroid.y),
            color="black",
            fontsize=9,
            ha="center",
            va="center",
            fontweight="bold",
            path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
        )
        if val >= 0.5:
            # ========= Country name slightly above =========
            ax2.annotate(
                row[AGGREGATION_LEVEL_2],
                (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
                color="black",
                fontsize=10,
                ha="center",
                va="bottom",
                fontweight="normal",
                path_effects=[pe.withStroke(linewidth=2, foreground="white",alpha=0.6)]
            )

# Add existing VREs to the plot

existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    committed_VREs_gdf=committed_VREs_plot[committed_VREs_plot['Technology'].str.lower()=='solar'],
                                                    committed_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=0.08,
                                                    marker_scale_committed=0.08,
                                                    marker_highlight_width=0.05)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    committed_VREs_gdf=committed_VREs_plot[committed_VREs_plot['Technology'].str.lower()=='wind'],
                                                    committed_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=0.08,
                                                    marker_scale_committed=0.08,
                                                    marker_highlight_width=0.05)

fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.35, 0.88), ncol=3, fontsize=8, frameon=False)

plt.tight_layout()
plt.savefig(vis_save_to_root/f"{country_kwd}_Capacity_by_Country.png", bbox_inches='tight', transparent=False)

In [ ]:
if solar_combined.crs != CRS_m:
    solar_cells_plot = solar_combined.to_crs(CRS_m)
else:    
    solar_cells_plot = solar_combined
if wind_combined.crs != CRS_m:
    wind_cells_plot = wind_combined.to_crs(CRS_m)
else:    
    wind_cells_plot = wind_combined

In [ ]:
no_land_patch = mpatches.Patch(
        facecolor="lightgray",
        edgecolor="gray",
        alpha=0.7,
        label="Economically unfeasible or no developable land",
    )

### Capacity

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5.5), dpi=1000)

# for double coulmn figure at 500 Dpi (72points per inch) minimum wide required is 3750 px, hence figsize=(7.5, 5) is recommended
# fig.suptitle(f"Resources for {country_name}", fontsize=18, fontweight='bold')

all_cells_gdf_plot.plot(ax=ax1, color="gray", edgecolor="none", alpha=0.7, zorder=1)
all_cells_gdf_plot.plot(ax=ax2, color="gray", edgecolor="none", alpha=0.7, zorder=1)

vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='solar',
                datafield='capacity',
                ax=ax1, 
                show=False)
vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='wind',
                datafield='capacity',
                ax=ax2, 
                show=False)

# Add existing VREs to the plot
existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=MARKER_SCALE_EXISTING,
                                                    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
                                                    marker_scale_committed=MARKER_SCALE_COMMITTED)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=MARKER_SCALE_EXISTING,
                                                    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
                                                    marker_scale_committed=MARKER_SCALE_COMMITTED)
# -------------------------------------------------
# Shared legend
# -------------------------------------------------
all_legends = []
all_legends.extend(solar_legends if solar_legends is not None else [])
all_legends.extend(wind_legends if wind_legends is not None else [])
# all_legends.append(no_land_patch) 

# de-duplicate legend labels
unique_handles = []
seen_labels = set()
for h in all_legends:
    label = h.get_label()
    if label not in seen_labels:
        unique_handles.append(h)
        seen_labels.add(label)

fig.legend(
    handles=unique_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.90),
    ncol=1,
    fontsize=9.5,
    frameon=False,
)
""" 
# -------------------------------------------------
# Plot note
# -------------------------------------------------
note_parts = []
note_parts.append(f"Solar LCOE threshold ≤ {solar_lcoe_threshold:.1f} \\$/MWh")
note_parts.append(f"Wind LCOE threshold ≤ {wind_lcoe_threshold:.1f} \\$/MWh")
note_parts.append("with capacity haircuts")
note_parts.append("with land-use competition")

plot_note = r" $\bullet$ ".join(note_parts)

fp = FontProperties(weight='bold', size=9.5)
fig.text(
    0.5,
    0.02,
    plot_note,
    ha="center",
    va="bottom",
    fontproperties=fp,
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="None",
        edgecolor="gray",
        alpha=0.9,
    ),
)

""" 
# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================
fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.5, 0.88), ncol=1, fontsize=8, frameon=False)
# Plot only boundaries (no fill)
aggregated_map_plot.boundary.plot(ax=ax1, color='#7A7A7A', linewidth=0.8,alpha=0.5, zorder=3)
aggregated_map_plot.boundary.plot(ax=ax2, color='k', linewidth=0.8, alpha=0.5,zorder=3)

""" 
# Add text annotations for capacity and country name
for idx, row in aggregated_map_plot.iterrows():
    centroid = row.geometry.centroid

    # Solar capacity on left
    ax1.annotate(
        f"{row['potential_capacity_solar_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax1.annotate(
        row[AGGREGATION_LEVEL_2],
        (centroid.x, centroid.y + ANNOTATION_ADJUSTMENT_METER),  # Adjust Y position for country name
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

    # Wind capacity on right
    ax2.annotate(
        f"{row['potential_capacity_wind_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax2.annotate(
        row[AGGREGATION_LEVEL_2],
        (centroid.x, centroid.y + ANNOTATION_ADJUSTMENT_METER),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

"""
# vis.add_compass_arrow_custom(ax1, text_offset=0.04)
vis.add_compass_arrow_custom(ax2, text_offset=0.04)
plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_combined_CAPACITY.png", bbox_inches='tight', transparent=True)

### Capacity Factor

* Individual Maps

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5.5), dpi=1000)
all_cells_gdf_plot.plot(ax=ax1, color="gray", edgecolor="none", alpha=0.7, zorder=1)
all_cells_gdf_plot.plot(ax=ax2, color="gray", edgecolor="none", alpha=0.7, zorder=1)

vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='solar',
                datafield='CF',
                cell_edge_color='white',
                cell_linewidth=0.2,
                ax=ax1, 
                show=False)
vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='wind',
                datafield='CF',
                cell_edge_color='white',
                cell_linewidth=0.2,
                ax=ax2, 
                show=False)

# Add existing VREs to the plot
existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    target_crs=CRS_m,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    committed_VREs_gdf=committed_VREs_plot[committed_VREs_plot['Technology'].str.lower()=='solar'],
                                                    committed_VRE_type_column='Technology',
                                                    marker_scale_existing=MARKER_SCALE_EXISTING,
                                                    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
                                                    marker_scale_committed=MARKER_SCALE_COMMITTED)

existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    committed_VREs_gdf=committed_VREs_plot[committed_VREs_plot['Technology'].str.lower()=='wind'],
                                                    committed_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                     marker_scale_existing=MARKER_SCALE_EXISTING,
                                                    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
                                                    marker_scale_committed=MARKER_SCALE_COMMITTED)

# -------------------------------------------------
# Shared legend
# -------------------------------------------------
all_legends = []
all_legends.extend(solar_legends if solar_legends is not None else [])
all_legends.extend(wind_legends if wind_legends is not None else [])
all_legends.append(no_land_patch)

# de-duplicate legend labels
unique_handles = []
seen_labels = set()
for h in all_legends:
    label = h.get_label()
    if label not in seen_labels:
        unique_handles.append(h)
        seen_labels.add(label)

fig.legend(
    handles=unique_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.90),
    ncol=1,
    fontsize=9.5,
    frameon=False,
)

# -------------------------------------------------
# Plot note
# -------------------------------------------------
note_parts = []
note_parts.append(f"Solar LCOE threshold ≤ {solar_lcoe_threshold:.1f} \\$/MWh")
note_parts.append(f"Wind LCOE threshold ≤ {wind_lcoe_threshold:.1f} \\$/MWh")
note_parts.append("with capacity haircuts")
note_parts.append("with land-use competition")

plot_note = r" $\bullet$ ".join(note_parts)

fp = FontProperties(weight='bold', size=9.5)
fig.text(
    0.5,
    0.02,
    plot_note,
    ha="center",
    va="bottom",
    fontproperties=fp,
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="None",
        edgecolor="gray",
        alpha=0.9,
    ),
)
# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================

# Plot only boundaries (no fill)
aggregated_map_plot.boundary.plot(ax=ax1, color='black', linewidth=0.7,alpha=0.7, zorder=3)
aggregated_map_plot.boundary.plot(ax=ax2, color='black', linewidth=0.7,alpha=0.7, zorder=3)

""" 
# Add text annotations for capacity and country name
for idx, row in aggregated_map_plot.iterrows():
    centroid = row.geometry.centroid
    cap = row["potential_capacity_solar_GW"]
    if cap == 0:
        continue  # Skip annotation for zero values
    else:
        # Solar capacity on left
        ax1.annotate(
            f"{row['potential_capacity_solar_GW']:.1f}",
            (centroid.x, centroid.y),
            color="black",
            fontsize=10,
            ha="center",
            va="center",
            fontweight="bold",
            zorder=4,
            path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
        )
        ax1.annotate(
            row[AGGREGATION_LEVEL_2],
            (centroid.x, centroid.y + ANNOTATION_ADJUSTMENT_METER),
            color="black",
            fontsize=9,
            ha="center",
            va="bottom",
            zorder=4,
            path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
        )

    # Wind capacity on right
    cap = row["potential_capacity_wind_GW"]
    if cap == 0:
        continue  # Skip annotation for zero values
    else:
        ax2.annotate(
            f"{row['potential_capacity_wind_GW']:.1f}",
            (centroid.x, centroid.y),
            color="black",
            fontsize=10,
            ha="center",
            va="center",
            fontweight="bold",
            zorder=4,
            path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
        )
        ax2.annotate(
            row[AGGREGATION_LEVEL_2],
            (centroid.x, centroid.y + ANNOTATION_ADJUSTMENT_METER),
            color="black",
            fontsize=9,
            ha="center",
            va="bottom",
            zorder=4,
            path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
        )
"""
# vis.add_compass_arrow_custom(ax1, text_offset=0.04)
vis.add_compass_arrow_custom(ax2, text_offset=0.04)
plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_combined_CF.png", bbox_inches='tight', transparent=True)

# Interactive Map

In [ ]:
# m = vis.make_lcoe_map(
#     wind_gdf=wind_cells,
#     solar_gdf=solar_cells,
#     sub_national_unit_tag=sub_national_unit_tag,
#     save_path=vis_save_to_root / f"{country_kwd}_lcoe_map_{RUN_ID}.html",
#     basemap_tiles="Esri WorldGrayCanvas",
#     wind_lcoe_max=wind_lcoe_threshold,
#     solar_lcoe_max=solar_lcoe_threshold,
# )

In [ ]:
# vis_save_to_root / f"{country_kwd}_lcoe_map_{RUN_ID}.html"

- plot thresholded map without haircut

## Clustering

### Cells

- Method 1 (simple)

In [ ]:
def get_country_clusters_cell( resource_type:str,
                              cells: pd.DataFrame=None,
                              lcoe_threshold: float=None,
                            ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Get country-level clusters of cells that meet the LCOE thresholds for solar and wind.
    Aggregates potential capacity, area, and costs for each resource by country.
    Returns two DataFrames: one for solar and one for wind, each with aggregated metrics.
    """
    
    if lcoe_threshold is not None:
        cells = cells[cells[f'lcoe_{resource_type}']<=lcoe_threshold].copy()
    else:
        cells = cells.copy()


    cells["Country"] = cells["Country"].apply(utils.standardize_tags)
    
    agg_dict = {
        f"total_capacity_{resource_type}_afterLandUseCompetition": "sum",
        f"capex_{resource_type}": "sum",
        f"fom_{resource_type}": "sum",
        f"vom_{resource_type}": "mean",
        f"grid_connection_cost_per_km_{resource_type}": "mean",
        f"tx_line_rebuild_cost_{resource_type}": "mean",
        f"Operational_life_{resource_type}": "mean",
        f"{resource_type}_CF_mean": "mean",
        f"lcoe_{resource_type}": "mean",
    }
    clusters = cells.groupby("Country").agg(agg_dict).reset_index()
    clusters[f"total_capacity_{resource_type}_afterLandUseCompetition"] = clusters[f"total_capacity_{resource_type}_afterLandUseCompetition"].round(2)
    clusters = clusters.rename(columns={ f"total_capacity_{resource_type}_afterLandUseCompetition": f"potential_capacity_{resource_type}_MW",
                                        f"capex_{resource_type}": f"capex_{resource_type}_MilUSDperMW",
                                        f"fom_{resource_type}": f"fom_{resource_type}_MilUSDperMW",
                                        f"vom_{resource_type}": f"vom_{resource_type}_MilUSDperMW",
                                        f"grid_connection_cost_per_km_{resource_type}": f"grid_connection_cost_per_km_{resource_type}_MilUSD",
                                        f"tx_line_rebuild_cost_{resource_type}": f"tx_line_rebuild_cost_{resource_type}_MilUSD",
                                        f"Operational_life_{resource_type}": f"operational_life_{resource_type}_years",
                                        f"{resource_type}_CF_mean": f"{resource_type}_CF_mean",
                                        f"lcoe_{resource_type}": f"lcoe_{resource_type}_USD_per_MWh"
                                        })
    clusters[f"potential_capacity_{resource_type}_GW"] = (clusters[f"potential_capacity_{resource_type}_MW"] / 1000).round(3)
    return clusters

- Method 2 (Standardized to absorb other clustering methods)

In [ ]:
# from RESource import cluster
# for resources in ['solar','wind']:
#     if resources == 'solar':
#         cells = solar_cells
#     else:
#         cells = wind_cells
#     cells["Country"] = cells["Country"].apply(utils.standardize_tags)
#     dataframe,optimal_k_df=cluster.pre_process_cluster_mapping(cells,'cluster_temp',1,'Country',resources)
#     optimal_k_df["Country"] = optimal_k_df["Country"].apply(utils.standardize_tags)
#     dataframe_filtered=dataframe[dataframe['Country'].isin(list(optimal_k_df['Country']))]
#     cluster_map,region_optimal_k_df = cluster.cells_to_cluster_mapping(cells_scored=cells, 
#                                                                    vis_directory='cluster_temp', 
#                                                                    wcss_tolerance=1,
#                                                                    sub_national_unit_tag='Country',
#                                                                    resource_type=resources,
#                                                                    sort_columns=[f'lcoe_{resources}', f'potential_capacity_{resources}']
#                                                                    )
#     cluster_map['Country'] = cluster_map['Country'].apply(utils.standardize_tags)
#     region_optimal_k_df['Country'] = region_optimal_k_df['Country'].apply(utils.standardize_tags)
#     cell_cluster_gdf, dissolved_indices = cluster.create_cells_Union_in_clusters(cluster_map, 
#                                                                                             region_optimal_k_df,
#                                                                                             'Country',
#                                                                                             resources)
#     cell_cluster_gdf['potential_capacity_GW'] = cell_cluster_gdf['potential_capacity']/1E3
#     if resources == 'solar':
#         solar_dissolved_indices=dissolved_indices
#         solar_clusters=cell_cluster_gdf#[['lcoe', 'potential_capacity_GW', 'CF_mean','Country','Cluster_no'] ]
#     else:
#         wind_dissolved_indices=dissolved_indices
#         wind_clusters=cell_cluster_gdf#[['lcoe', 'potential_capacity_GW', 'CF_mean','Country','Cluster_no'] ]

In [ ]:
solar_clusters= get_country_clusters_cell(resource_type='solar', cells=solar_combined, lcoe_threshold=solar_lcoe_threshold)
wind_clusters= get_country_clusters_cell(resource_type='wind', cells=wind_combined, lcoe_threshold=wind_lcoe_threshold)

In [ ]:
# Map Country -> Country_code
solar_clusters["Country_code"] = solar_clusters["Country"].map(region_to_code)
solar_clusters=solar_clusters.set_index(solar_clusters["Country_code"])
solar_clusters.to_csv(results_save_to_root/"solar_clusters.csv", index=False)
wind_clusters["Country_code"] = wind_clusters["Country"].map(region_to_code)
wind_clusters=wind_clusters.set_index(wind_clusters["Country_code"])
wind_clusters.to_csv(results_save_to_root/"wind_clusters.csv", index=False)
print(f"{results_save_to_root/'wind_clusters.csv'} and {results_save_to_root/'solar_clusters.csv'} saved successfully.")

### Timeseries

In [ ]:
solar_ts_dict:dict[pd.DataFrame]=[combined_store[region]['timeseries_solar'] for region in combined_store]
wind_ts_dict:dict[pd.DataFrame]=[combined_store[region]['timeseries_wind'] for region in combined_store]

In [ ]:
solar_ts=pd.DataFrame()
wind_ts=pd.DataFrame()
for resource_type in ['solar','wind']:
    for region in combined_store.keys():
        ts_df=combined_store[region][f'timeseries_{resource_type}']
        ts_mean=ts_df.mean(axis=1) # to check if there is any non-zero value, if all values are zero then it means timeseries data is not available for that region
        ts_mean['Country']=region
        ts_mean.index.name="timestamp"
        if resource_type == 'solar':
            solar_ts[region]=ts_mean

        else:
            wind_ts[region]=ts_mean
            
            
wind_ts.to_csv(results_save_to_root/f"wind_timeseries.csv", index=True)
print(f"Saved wind timeseries to {results_save_to_root/f'wind_timeseries.csv'}")
solar_ts.to_csv(results_save_to_root/f"solar_timeseries.csv", index=True)
print(f"Saved solar timeseries to {results_save_to_root/f'solar_timeseries.csv'}")

#### plot

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

def plot_ts_dashboard(ts_df: pd.DataFrame, resource: str = "solar"):
    """
    Interactive dashboard for hourly CF timeseries by country.

    Parameters:
        ts_df: DataFrame with datetime index and country columns (e.g. AL, BA, ...).
        resource: Label for titles ("solar" or "wind").
    """
    # Drop any non-numeric rows (e.g. the 'Country' label row at the bottom)
    ts_df = ts_df.apply(pd.to_numeric, errors="coerce").dropna()
    ts_df.index = pd.to_datetime(ts_df.index)

    daily_mean = ts_df.resample("D").mean()
    countries = ts_df.columns.tolist()

    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(
            f"Hourly {resource.title()} CF — All Countries",
            f"Daily Mean {resource.title()} CF — All Countries",
            f"Mean Diurnal Profile (Hour of Day)",
        ),
        vertical_spacing=0.08,
        row_heights=[0.4, 0.35, 0.25],
    )

    # --- Row 1: Hourly profiles ---
    for country in countries:
        fig.add_trace(
            go.Scatter(
                x=ts_df.index,
                y=ts_df[country],
                name=country,
                legendgroup=country,
                mode="lines",
                opacity=0.7,
                line=dict(width=0.8),
            ),
            row=1, col=1,
        )

    # --- Row 2: Daily mean ---
    for country in countries:
        fig.add_trace(
            go.Scatter(
                x=daily_mean.index,
                y=daily_mean[country],
                name=country,
                legendgroup=country,
                showlegend=False,
                mode="lines",
                line=dict(width=1.2),
            ),
            row=2, col=1,
        )

    # --- Row 3: Mean diurnal profile ---
    diurnal = ts_df.groupby(ts_df.index.hour).mean()
    for country in countries:
        fig.add_trace(
            go.Scatter(
                x=diurnal.index,
                y=diurnal[country],
                name=country,
                legendgroup=country,
                showlegend=False,
                mode="lines+markers",
                line=dict(width=2),
            ),
            row=3, col=1,
        )

    fig.update_xaxes(title_text="Time", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_xaxes(title_text="Hour of Day", dtick=1, row=3, col=1)

    fig.update_yaxes(title_text="CF", row=1, col=1)
    fig.update_yaxes(title_text="CF (daily mean)", row=2, col=1)
    fig.update_yaxes(title_text="CF (mean)", row=3, col=1)

    fig.update_layout(
        height=1000,
        title_text=f"{resource.title()} Capacity Factor Dashboard",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
        hovermode="x unified",
    )

    fig.show()
    fig.write_html(vis_save_to_root/f"{resource}_CF_dashboard.html")


# --- Usage ---
# plot_ts_dashboard(wind_ts_df, resource="wind")
# plot_ts_dashboard(solar_ts_df, resource="solar")

In [ ]:
plot_ts_dashboard(wind_ts, resource="wind")
plot_ts_dashboard(solar_ts, resource="solar")